In [1]:
from tkinter import *
import math

# CONSTANTS

In [2]:
PINK = "#e2979c"
RED = "#e7305b"
GREEN = "#9bdeac"
YELLOW = "#f7f5dd"
FONT_NAME = "Courier"
WORK_MIN = 25
SHORT_BREAK_MIN = 5
LONG_BREAK_MIN = 20

In [3]:
reps = 0

# 218 TIMER MECHANISM & COUNTDOWN MECHANISM

## 218.1 My Version

主要的麻烦就是要不断地点击start button来开始任意一个小session，也就是不能自动调用自己。

In [4]:
# ---------------------------- TIMER MECHANISM ------------------------------- # 
def start_timer_myver():
    global reps     # 如果不写这一句，会报错：UnboundLocalError: cannot access local variable 'reps' where it is not associated with a value
    print(reps)
    work_sec = WORK_MIN
    short_break_sec = SHORT_BREAK_MIN
    long_break_sec = LONG_BREAK_MIN

    # If it's the 1st/3rd/5th/7th reps
    if reps % 2 == 1:
        count_down_myver(work_sec)
        title_label.config(text="Work", fg=GREEN)
        reps += 1
    # If it's the 8th rep:
    elif reps % 8 == 0:     # 原本甚至写的是 reps == 8，因为我就没有考虑过完成了一个大session后该怎么办
        count_down_myver(long_break_sec)
        title_label.config(text="Break", fg=RED)
        reps +=1
    # If it's 2nd/4th/6th rep:
    else:
        count_down_myver(short_break_sec)
        title_label.config(text="Break", fg=PINK)
        reps += 1

# ---------------------------- COUNTDOWN MECHANISM ------------------------------- # 
def count_down_myver(count):
    global reps
    count_min = math.floor(count / 60)
    count_sec = count % 60
    if count_sec < 10:
        # count_sec = "0" + str(count_sec)
        count_sec = f"0{count_sec}"

    canvas.itemconfig(timer_text, text=f"{count_min}:{count_sec}")

    if count > 0:  # count = 0时，停止递归（使用递归时需要设置终止语句）
        window.after(1000, count_down, count - 1)  # 1000 毫秒后，调用 count_down(count - 1)

## 218.2 Teacher's Version
主要好处：在数完一个session后，可以直接进入下一个session，不像我的版本那样需要人为点击"Start" button。而且reps和两个函数`start_timer`, `count_down`之间的逻辑耦合也更强。

比如说，reps数到8后，可以丝滑地进入9，开始下一个大session，这是可以不需要任何多余操作直接进入的。

In [5]:
# ---------------------------- TIMER MECHANISM ------------------------------- # 
def start_timer():
    global reps     # 如果不写这一句，会报错：UnboundLocalError: cannot access local variable 'reps' where it is not associated with a value
    reps +=1
    print(reps)

    work_sec = WORK_MIN * 60
    short_break_sec = SHORT_BREAK_MIN * 60
    long_break_sec = LONG_BREAK_MIN * 60

    # If it's the 1st/3rd/5th/7th reps
    if reps % 2 == 1:
        count_down(work_sec)
        title_label.config(text="Work", fg=GREEN)
    # If it's the 8th rep:
    elif reps % 8 == 0:
        count_down(long_break_sec)
        title_label.config(text="Break", fg=RED)
    # If it's 2nd/4th/6th rep:
    else:
        count_down(short_break_sec)
        title_label.config(text="Break", fg=PINK)

# ---------------------------- COUNTDOWN MECHANISM ------------------------------- # 
def count_down(count):
    global reps
    count_min = math.floor(count / 60)
    count_sec = count % 60
    if count_sec < 10:
        # count_sec = "0" + str(count_sec)
        count_sec = f"0{count_sec}"

    canvas.itemconfig(timer_text, text=f"{count_min}:{count_sec}")

    if count > 0:  # count = 0时，停止递归（使用递归时需要设置终止语句）
        window.after(1000, count_down, count - 1)  # 1000 毫秒后，调用 count_down(count - 1)
    else:
        start_timer()

### 注意观察老师写的这个`start_timer`函数
从这个函数来看，最后面的那个`if-else`分支，进来之后，先是做`count_down`，尤其`count_down`自己还是个递归的逻辑，那么应该一直在这个里面走，countdown完毕才会跳出来把`title_label`的文本和颜色给改了吧？

但是实际运行的时候，一旦进入if-else的对应分支，`title_label`的文本和颜色就立马变了，而且倒计时也是同步开始的。这是咋回事？

这是一个计算机进程的问题吗？

#### (1) 这个流程到底是怎么执行的？
这是 GUI 程序最反直觉的地方：你以为 `count_down()` 会像普通函数一样 “一直运行到全部指令都结束才返回”，但实际上它不会。它只是更新一次界面，然后预约下一次调用，接着立刻返回。

关键就在最后面的这个if-else分支：
```python
if count > 0:
    window.after(1000, count_down, count - 1)
else:
    start_timer()
```

其中的 `window.after(1000, count_down, count - 1)` 并不是等 1000ms，然后继续执行 `count_down(count - 1)` 它真正做的是把 *“ `1000`ms 之后调用 `count_down(count - 1)`”* 这件事登记给 `Tkinter`，然后当前函数立刻结束。

所以当程序进入这个分支时：

```python
if reps % 2 == 1:
    count_down(work_sec)
    title_label.config(text="Work", fg=GREEN)
```

真正的执行顺序是：

```
进入 if 分支
→ 调用 count_down(work_sec)
→ count_down 立刻把 timer_text 改成当前时间，比如 10:00
→ count_down 用 window.after(...) 预约 1 秒后再调用自己
→ count_down 立刻返回
→ 执行 title_label.config(text="Work", fg=GREEN)
→ start_timer() 结束
→ 程序回到 window.mainloop()
→ Tkinter 开始处理界面刷新、按钮响应、after 预约任务
→ 1 秒后，Tkinter 再调用 count_down(work_sec - 1)
```

所以你看到的现象就是：标题立刻变成 Work/Break，倒计时也立刻开始。因为 `count_down()` 并没有“卡在里面等倒计时结束”，它只是安排了下一次倒计时。

可以把它理解成 “每次 `count_down()` 只负责一帧”：
```
第 1 次 count_down：显示 10:00，预约下一帧
第 2 次 count_down：显示 09:59，预约下一帧
第 3 次 count_down：显示 09:58，预约下一帧
……
倒到 0：不再预约下一帧，改为调用 start_timer()
```

#### (2) 这个`count_down`的逻辑是有点像递归，但并不是普通的递归

普通递归是这样：
```python
def f(n):
    f(n - 1)
```
这种是立刻调用自己，函数会一层压一层，直到结束才一层层返回。

但 Tkinter 这里是：
```python
window.after(1000, count_down, count - 1)
```
它是延迟调用自己，不是现在调用自己。当前这一层 `count_down()` 很快就结束了，下一次是 1000ms 后由 `Tkinter` 的事件循环重新调用。它不会一直占着*函数调用栈*不出来。

所以更准确地说，这不是传统意义上的递归，而是用 `after()` 实现的定时回调循环。


#### (3) 概念：事件循环 event loop / 回调 callback

这不是进程问题，也通常不是多线程问题。

这里主要是 事件循环 event loop / 回调 callback 的问题。

`Tkinter` 一般是在一个主线程里跑的，`mainloop()` 就像一个调度员：它负责监听按钮点击、刷新窗口、执行 `after()` 预约好的函数。`after()` 不是开了一个新进程，也不是让另一个线程偷偷倒计时，而是把任务排进 Tkinter 的事件队列里，等时间到了再由主循环处理。

#### 这就是 GUI 代码和普通顺序代码最不一样的地方：代码看起来写在一起，但真正的执行节奏是由事件循环安排的。

# 215. UI SETUP

In [6]:
# Window
window = Tk()
window.title("Pomodoro")
window.config(padx=100, pady=50, bg=YELLOW)

## Label1
title_label = Label(text="Timer", font=(FONT_NAME, 50), fg=GREEN, bg=YELLOW)
title_label.grid(column=1, row=0)

## Canvas
canvas = Canvas(width=200, height=224, bg=YELLOW, highlightthickness=0)
tomato_img = PhotoImage(file="tomato.png")
canvas.create_image(100, 112, image=tomato_img)
timer_text = canvas.create_text(100, 130, text="00:00", fill="white", font=(FONT_NAME, 35, "bold"))
canvas.grid(column=1, row=1)

## Button1
start_button = Button(text="Start", highlightthickness=0, command=start_timer)
start_button.grid(column=0, row=2)

## Button2
reset_button = Button(text="Reset", highlightthickness=0)
reset_button.grid(column=2, row=2)

# Label2
check_marks = Label(text="✔", fg=GREEN, bg=YELLOW)
check_marks.grid(column=1, row=3)

window.mainloop()

1
